# Mini Semantic Search

## 1. Project Overview

This project builds a mini semantic search system using customer support question-and-answer data.

Unlike traditional keyword search, semantic search aims to find relevant results based on the meaning of a query, even when the wording differs from the stored text.

We will use Pandas and NumPy for data preparation, a text embedding model to represent text as numerical vectors, and ChromaDB to store and retrieve relevant results.


## 2. Load the Dataset

We will use Pandas' `read_csv()` function to load our customer support dataset into a DataFrame. A DataFrame organizes the data into rows and columns, allowing us to inspect, clean, and prepare the questions and answers for semantic search.

In [26]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv")

print(df.shape)
print(df.head())

(26872, 5)
   flags                                        instruction category  \
0      B   question about cancelling order {{Order Number}}    ORDER   
1    BQZ  i have a question about cancelling oorder {{Or...    ORDER   
2   BLQZ    i need help cancelling puchase {{Order Number}}    ORDER   
3     BL         I need to cancel purchase {{Order Number}}    ORDER   
4  BCELN  I cannot afford this order, cancel purchase {{...    ORDER   

         intent                                           response  
0  cancel_order  I've understood you have a question regarding ...  
1  cancel_order  I've been informed that you have a question ab...  
2  cancel_order  I can sense that you're seeking assistance wit...  
3  cancel_order  I understood that you need assistance with can...  
4  cancel_order  I'm sensitive to the fact that you're facing f...  


### 2.2 Understanding the Dataset

Our dataset contains customer support requests and their corresponding responses. Before cleaning the data, we will examine its columns, data types, and overall structure.

Understanding these details will help us identify which columns are useful for semantic search and whether any data needs to be cleaned.

In [27]:
print("Dataset dimensions:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

Dataset dimensions: (26872, 5)

Column names:
['flags', 'instruction', 'category', 'intent', 'response']

Data types:
flags          str
instruction    str
category       str
intent         str
response       str
dtype: object

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 26872 entries, 0 to 26871
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   flags        26872 non-null  str  
 1   instruction  26872 non-null  str  
 2   category     26872 non-null  str  
 3   intent       26872 non-null  str  
 4   response     26872 non-null  str  
dtypes: str(5)
memory usage: 1.0 MB


### 2.3 Checking Data Quality

Before preparing our text for semantic search, we need to check for duplicate records, empty values, and whitespace-only text.

We will use Pandas to identify these issues and NumPy to help summarize the results. We will inspect the data first rather than immediately deleting anything.

In [28]:
# Count duplicate rows
duplicate_count = df.duplicated().sum()

# Count missing values in each column
missing_values = df.isna().sum()

# Count empty or whitespace-only strings in each column
blank_values = df.apply(
    lambda column: column.str.strip().eq("").sum()
)

print("Duplicate rows:", duplicate_count)

print("\nMissing values per column:")
print(missing_values)

print("\nBlank strings per column:")
print(blank_values)

Duplicate rows: 0

Missing values per column:
flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

Blank strings per column:
flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64


### 2.4 Selecting Relevant Columns

Our goal is to find customer support answers that are relevant to a user's question. The `instruction` column contains customer requests, while the `response` column contains the corresponding answers.

We will create a function called `prepare_data()` to select these columns and rename them to `question` and `answer`. Keeping this operation inside a function makes it reusable and keeps our notebook organized.

In [29]:
def prepare_data(df):
    # Select the columns needed for semantic search
    faq_df = df[["instruction", "response"]].copy()

    # Rename the columns
    faq_df = faq_df.rename(
        columns={
            "instruction": "question",
            "response": "answer"
        }
    )

    return faq_df


faq_df = prepare_data(df)

print("Prepared dataset shape:", faq_df.shape)
display(faq_df.head())

Prepared dataset shape: (26872, 2)


,question,answer
0,question about cancelling order {{Order Number}},I've understood you have a question regarding ...
1,i have a question about cancelling oorder {{Or...,I've been informed that you have a question ab...
2,i need help cancelling puchase {{Order Number}},I can sense that you're seeking assistance wit...
3,I need to cancel purchase {{Order Number}},I understood that you need assistance with can...
4,"I cannot afford this order, cancel purchase {{...",I'm sensitive to the fact that you're facing f...


### 2.5 Text Preprocessing

Text preprocessing helps make our dataset more consistent before semantic search. We will define a `clean_text()` function that converts text to lowercase and removes unnecessary leading, trailing, and repeated whitespace.

We will preserve punctuation and the original wording because they may carry useful meaning. We will then apply the function to both questions and answers.

In [30]:
def clean_text(text):
    # Convert text to lowercase
    text = text.lower()

    # Remove unnecessary whitespace
    text = " ".join(text.split())

    return text


# Clean questions and answers
faq_df["question"] = faq_df["question"].apply(clean_text)
faq_df["answer"] = faq_df["answer"].apply(clean_text)

display(faq_df.head())

,question,answer
0,question about cancelling order {{order number}},i've understood you have a question regarding ...
1,i have a question about cancelling oorder {{or...,i've been informed that you have a question ab...
2,i need help cancelling puchase {{order number}},i can sense that you're seeking assistance wit...
3,i need to cancel purchase {{order number}},i understood that you need assistance with can...
4,"i cannot afford this order, cancel purchase {{...",i'm sensitive to the fact that you're facing f...
